# Prior exploration

Draws samples from the Bilaplacian SPDE prior on the fiber-angle field, to get a feeling for the hyperparameters $\kappa$ (correlation length) and $\tau$ (marginal variance). Change `KAPPA`, `TAU` (and `SEED`) in the parameter cell and re-run from there; only the prior is built, so no eikonal solver or data is involved.

The prior mean is zero, so a sample is the pure prior fluctuation. Samples are re-branched onto a single $\pi$-periodic branch around their axial mean, as for the synthetic ground truth in preprocessing. Only the mesh of the example patient (`example_data/raw/mesh.vtu`) is loaded.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv

from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
from bayes_cep.mesh.io import create_dolfinx_mesh, load_pyvista_mesh
from bayes_cep.posterior.prior import PriorSettings, build_fiber_angle_prior
from bayes_cep.statistics.axial_statistics import (
    compute_axial_mean_and_variance,
    shift_angles_to_minimize_axial_variance,
)
from bayes_cep.statistics.correlation_length import (
    CorrelationLengthSettings,
    estimate_correlation_length,
)

pv.set_jupyter_backend("trame")

DATA_DIR = Path.cwd().parent / "example_data"
mesh = load_pyvista_mesh(DATA_DIR / "raw" / "mesh.vtu")

interpolation_strategy = NearestNeighborInterpolationStrategy()
connectivity = mesh.cells.reshape(-1, 4)[:, 1:]
vertex_to_simplex_matrix = interpolation_strategy.assemble_matrix(mesh.points, connectivity)
prior_mean_angle_field = np.zeros(mesh.n_points)
dlx_mesh = create_dolfinx_mesh(mesh)


def draw_sample():
    # Same re-branching onto a single pi-periodic branch as the synthetic ground truth
    return shift_angles_to_minimize_axial_variance(prior.generate_sample()).flatten()

## Parameters

In [ ]:
KAPPA = 0.05
TAU = 10.0
SEED = 0
NUM_SAMPLES = 4  # number of samples shown in the grid below

## Build the prior

The seed fixes the sequence of draws; re-running this cell restarts it.

In [ ]:
prior = build_fiber_angle_prior(
    dlx_mesh,
    PriorSettings(mean_vector=prior_mean_angle_field, kappa=KAPPA, tau=TAU, seed=SEED),
)

## One sample

In [ ]:
sample = draw_sample()
print(f"sample: std {sample.std():.3f} rad, min {sample.min():.3f}, max {sample.max():.3f}")

plot_mesh = mesh.copy()
plot_mesh.cell_data["angle"] = vertex_to_simplex_matrix @ sample
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv")
plotter.add_text(f"prior sample (kappa={KAPPA}, tau={TAU})")
plotter.show()

## Several samples

Distribution of the sample values over several draws, to judge the marginal scale beyond a single realisation.

In [ ]:
samples = np.stack([draw_sample() for _ in range(NUM_SAMPLES)])
print(f"pointwise std over {NUM_SAMPLES} samples: mean {samples.std(axis=0).mean():.3f} rad")

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(samples.ravel(), bins=60, density=True)
ax.set_xlabel("sample value [rad]")
ax.set_ylabel("density")
fig.tight_layout()

plotter = pv.Plotter(shape=(1, NUM_SAMPLES))
for i, values in enumerate(samples):
    plotter.subplot(0, i)
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["angle"] = vertex_to_simplex_matrix @ values
    plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv", show_scalar_bar=False)
    plotter.add_text(f"sample {i + 1}", font_size=8)
plotter.show()

## Pointwise variance and correlation length

Axial statistics of many prior samples (`NUM_STAT_SAMPLES`; a few hundred are needed for a smooth estimate): the pointwise axial variance, and the correlation distance (see `bayes_cep.statistics.correlation_length`). For the latter, the axial correlation is pooled over pairs of vertices and binned by graph distance along the mesh; the primary output is the model-free distance where the binned correlation first drops below $1/e$. A Matérn ($\nu = 1$) fit of the binned curve is reported as a secondary, effective length. `MAX_DISTANCE` should be a few times the expected correlation length (in mesh units), and `BOUNDARY_MARGIN` keeps base vertices away from the mesh boundary.

In [ ]:
NUM_STAT_SAMPLES = 300
MAX_DISTANCE = 50
BOUNDARY_MARGIN = 10.0

stat_samples = np.stack([draw_sample() for _ in range(NUM_STAT_SAMPLES)])
_, axial_variance = compute_axial_mean_and_variance(stat_samples)

plot_mesh = mesh.copy()
plot_mesh.point_data["axial std"] = np.sqrt(axial_variance)
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="axial std", cmap="viridis")
plotter.add_text("pointwise axial standard deviation [rad]")
plotter.show()

correlation = estimate_correlation_length(
    mesh.points,
    connectivity,
    stat_samples,
    CorrelationLengthSettings(max_distance=MAX_DISTANCE, boundary_margin=BOUNDARY_MARGIN),
)
print(f"correlation distance (drops below 1/e): {correlation.correlation_distance_threshold:.2f}")
print(f"fitted Matérn length (secondary):        {correlation.length:.2f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(correlation.bin_distances, correlation.bin_correlations, "o", label="binned")
ax.plot(correlation.bin_distances, correlation.fitted_correlations, label="Matérn fit")
ax.set_xlabel("graph distance")
ax.set_ylabel("axial correlation")
ax.legend()
fig.tight_layout()